# Diabetes status classification
Exploratory revised notebook, November 2026. **This notebook has not been executed end to end.** Outputs are cleared. The current verified model-comparison entry point is [scripts/reproduce_analysis.py](../scripts/reproduce_analysis.py), with separately recorded aggregate results.
The ordinary-SMOTE comparison below retains numeric-coded categories and repeated exploratory holdout inspection. Use the current CLI for explicit categorical handling and training-only model selection. This previously examined holdout cannot become an independent test through rerunning. Optional permutation/SHAP sections below have not been verified by clean execution.
See [the original executed notebook](archive/Final_original.ipynb), [findings](../docs/FINDINGS.md) and [reproduction instructions](../docs/REPRODUCIBILITY.md).
This academic group project classifies a survey-derived yes/borderline label; it does not forecast future onset or provide medical advice.


In [ ]:
# MATH 3310 group project: survey-derived diabetes status classification
# This revised notebook separates historical evidence from current code.
# Core workflow: preparation, stratified validation, LR/RF comparison and imbalance handling.
# Optional SHAP interpretation follows model evaluation.
# Documentation and original outputs are linked in the opening Markdown cell.


In [ ]:
# Import libraries

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import seaborn as sns

from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV, KFold, cross_val_predict
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score, ConfusionMatrixDisplay
from sklearn.feature_selection import SelectFromModel
from sklearn.tree import DecisionTreeClassifier

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.metrics import geometric_mean_score as gmean_score

In [ ]:
def evaluate_classifier(model, X_train, X_test, y_train, y_test, target_name=""):
    """Fit and evaluate. Test results must not drive subsequent model selection."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    roc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]) if hasattr(model, "predict_proba") else np.nan
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "positive_f1": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "gmean": gmean_score(y_test, y_pred),
        "roc_auc": roc,
    }
    print(target_name, metrics)
    print(classification_report(y_test, y_pred, zero_division=0))
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    print("Confusion matrix, rows actual and columns predicted [0, 1]:", cm)
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=[0, 1],
        display_labels=["Reported no", "Reported yes/borderline"], cmap="Blues", ax=ax)
    ax.set_title(target_name)
    fig.tight_layout()
    filename = target_name.replace(" ", "_").replace("/", "_").lower() + "_confusion_matrix.png"
    fig.savefig(FIGURE_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()
    return y_pred, metrics


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "diabetes_analysis.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the cloned Diabetes_Prediction repository.")
sys.path.insert(0, str(ROOT))
from diabetes_analysis import activity_code, diabetes_label
FIGURE_DIR = ROOT / "reports" / "generated"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
file_path = ROOT / "data" / "raw" / "NHANES_age_prediction.csv"
if not file_path.is_file():
    raise FileNotFoundError(f"Missing original prepared input: {file_path}. See docs/REPRODUCIBILITY.md.")
df = pd.read_csv(file_path)
required = {"SEQN", "RIDAGEYR", "RIAGENDR", "PAQ605", "BMXBMI", "LBXGLU", "DIQ010", "LBXGLT", "LBXIN", "age_group"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Input lacks required columns: {sorted(missing)}")
print("Input shape:", df.shape)


In [ ]:
# Display information about the DataFrame
print(df.info())

In [ ]:
# Rename columns for clarity and check for missing values

df.rename(columns={
    'SEQN': 'ID',
    'RIDAGEYR': 'Age',
    'RIAGENDR': 'Gender',
    'PAQ605': 'Physical_Activity',
    'BMXBMI': 'BMI',
    'LBXGLU': 'Glucose_Level',
    'DIQ010': 'Diabetes_Status',
    'LBXGLT': 'Oral_Glucose_Test',
    'LBXIN': 'Insulin_Level'}, inplace=True)


print(df.isnull().sum())
print("Age group missing:", df["age_group"].isnull().sum())

In [ ]:
# CDC PAQ605: 7=refused, 9=do not know. Neither means no.
# This corrects the original 7 -> 2 recode; historical outputs are not revised results.
df["Physical_Activity"] = df["Physical_Activity"].map(activity_code)
# Project target choice: yes and borderline together; exclude other responses.
labels = df["Diabetes_Status"].map(diabetes_label)
diab_df = df.loc[labels.notna()].copy()
diab_df["diabetes_binary"] = labels.loc[labels.notna()].astype(int)


In [ ]:
# check for impossible negative values in the df (for integers and floats in df)
for col in df.select_dtypes(include=['int64', 'float64']).columns:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f"Column '{col}' has {neg_count} impossible negative values.")
    else:
        print(f"Column '{col}' has no impossible negative values.")

In [ ]:
print(diab_df["diabetes_binary"].value_counts(), "\n")
print("Class proportions (diabetes_binary):")
print(diab_df["diabetes_binary"].value_counts(normalize=True), "\n")

In [ ]:
# Diabetes-risk classification (binary label from Diabetes Status)
# we are going to focus on age, sex, BMI, glucose-related features, and vigorous exercise
# dropping diabetes ID, Insulin_Level, and age_group

X_diab = diab_df.drop(columns=['Diabetes_Status','age_group', 'diabetes_binary', 'ID', 'Insulin_Level'])
y_diab = diab_df['diabetes_binary']

In [ ]:
X_diab.head()

In [ ]:
# Set a clean, modern style
sns.set_style("whitegrid")
plt.figure(figsize=(8, 6))

# Create the countplot with a colorblind-friendly palette
ax = sns.countplot(
    x=diab_df['diabetes_binary'],
    palette="Set2"  # or try "colorblind" for full accessibility
)

# Fix title and axis labels to reflect actual content
plt.title('Class Distribution of Diabetes Status', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Diabetes Status (0 = Reported no, 1 = Reported yes/borderline)', fontsize=12)
plt.ylabel('Number of Individuals', fontsize=12)

# Add count and percentage annotations on top of bars
total = len(diab_df)
for p in ax.patches:
    count = int(p.get_height())
    percentage = 100 * count / total
    ax.annotate(f'{count}\n({percentage:.1f}%)', 
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11, fontweight='medium')

# Improve layout and spacing
plt.tight_layout()

# Save with high DPI for publication quality
plt.savefig(FIGURE_DIR / "diabetes_class_distribution_300dpi.png", dpi=300, bbox_inches='tight')

# Display
plt.show()

In [ ]:
# Set style for publication-quality plots
sns.set_style("whitegrid")
plt.figure(figsize=(10, 8))

# Create pairplot with hue = diabetes_binary
g = sns.pairplot(
    diab_df[['BMI', 'Age', 'Glucose_Level', 'Oral_Glucose_Test', 'diabetes_binary']],
    hue='diabetes_binary',
    palette=['#66c2a5', '#fc8d62'],  # Colorblind-friendly: green for No Diabetes, orange for Diabetes
    diag_kind='kde',                 # Density plots on diagonal
    plot_kws={'alpha': 0.6, 's': 40}, # Point transparency and size
    diag_kws={'fill': True, 'bw_adjust': 0.75}  # Smoother density curves
)

# Customize titles and labels
g.fig.suptitle('Pairwise Relationships by Diabetes Status', fontsize=16, fontweight='bold', y=0.98)
for ax in g.axes.flat:
    if ax.get_xlabel():
        ax.set_xlabel(ax.get_xlabel().replace('_', ' ').title(), fontsize=11)
    if ax.get_ylabel():
        ax.set_ylabel(ax.get_ylabel().replace('_', ' ').title(), fontsize=11)

# Improve legend
g._legend.set_title('Diabetes Status')
g._legend.get_texts()[0].set_text('No Diabetes (0)')
g._legend.get_texts()[1].set_text('Yes/borderline (1)')

# Save high-res figure
filename = "pairplot_diabetes_features_300dpi.png"
plt.savefig(FIGURE_DIR / filename, dpi=300, bbox_inches='tight')

# Show plot
plt.show()

In [ ]:
# Define features of interest
features = ['BMI', 'Age', 'Glucose_Level', 'Oral_Glucose_Test']

# Melt the DataFrame for easy plotting
df_melted = diab_df.melt(
    id_vars='diabetes_binary',
    value_vars=features,
    var_name='Feature',
    value_name='Value'
)

# Map feature names to human-readable labels
label_map = {
    'BMI': 'BMI',
    'Age': 'Age (years)',
    'Glucose_Level': 'Fasting Glucose (mg/dL)',
    'Oral_Glucose_Test': 'OGTT (mg/dL)'
}
df_melted['Feature'] = df_melted['Feature'].map(label_map)

# Set style
sns.set_style("whitegrid")
plt.figure(figsize=(10, 6))

# Create grouped boxplot
ax = sns.boxplot(
    data=df_melted,
    x='Feature',
    y='Value',
    hue='diabetes_binary',
    palette=['#66c2a5', '#fc8d62'],  # Colorblind-friendly: No Diabetes / Diabetes
    width=0.7,
    fliersize=3
)

# Customize legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, ['Reported no', 'Reported yes/borderline'], title='Diabetes Status', fontsize=11, title_fontsize=12)

# Labels and title
plt.title('Distribution of Key Features by Diabetes Status', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('')
plt.ylabel('Value', fontsize=12)
plt.xticks(rotation=0, fontsize=11)
plt.tight_layout()

# Save high-resolution figure
filename = "boxplot_features_by_diabetes_300dpi.png"
plt.savefig(FIGURE_DIR / filename, dpi=300, bbox_inches='tight')

# Show
plt.show()


In [ ]:
# Optional problexity analysis remains in the archived original notebook.
# It is not used for model selection or required by this revised workflow.


In [ ]:
# See the archived original for the historical complexity figure.


In [ ]:
# train/test split the data

X_train_diab, X_test_diab, y_train_diab, y_test_diab = train_test_split(X_diab, y_diab, test_size=0.2, stratify=y_diab, random_state=1117)
print("Training set class distribution:")
print(y_train_diab.value_counts(normalize=True))
print("\nTesting set class distribution:")
print(y_test_diab.value_counts(normalize=True))

In [ ]:
# clarify numerical and categorical data (even though it's clear that we have no categorical data here)

diab_numerical_features = X_diab.select_dtypes(include=["int64", "float64"]).columns.tolist()
diab_categorical_features = X_diab.select_dtypes(include=["object", "category"]).columns.tolist()

print("Diabetes numeric features:", diab_numerical_features)
print("Diabetes categorical features:", diab_categorical_features, "\n")

In [ ]:
# create preprocessing pipelines for numerical and categorical data

diab_numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
diab_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)
diab_preprocessor = ColumnTransformer(
    transformers=[
        ("num", diab_numerical_transformer, diab_numerical_features),
        ("cat", diab_categorical_transformer, diab_categorical_features),
    ]
)
cv_strat_diab = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

diab_preprocessor

In [ ]:
# baseline  logistic regression and random forest models for diabetes classification

diab_log_pipe = Pipeline([
    ("preprocessor", diab_preprocessor),
    ("classifier", LogisticRegression(solver="liblinear", max_iter=1000))
])

diab_rf_pipe = Pipeline([
    ("preprocessor", diab_preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))
])

In [ ]:
# CV for baseline Logistic Regression

log_cv_scores_diab = cross_val_score(
    diab_log_pipe,
    X_train_diab,
    y_train_diab,
    cv=cv_strat_diab,
    scoring="balanced_accuracy"
)
rf_cv_scores_diab = cross_val_score(
    diab_rf_pipe, 
    X_train_diab,
    y_train_diab, 
    cv=cv_strat_diab, 
    scoring="balanced_accuracy"
)

_ , diab_log_metrics = evaluate_classifier(diab_log_pipe, X_train_diab, X_test_diab, y_train_diab, y_test_diab, target_name="Diabetes (Logistic Regression - baseline)")
_ , diab_rf_metrics  = evaluate_classifier(diab_rf_pipe, X_train_diab, X_test_diab, y_train_diab, y_test_diab, target_name="Diabetes (Random Forest - baseline)")

In [ ]:
# hyperparameter tuning with GridSearchCV for Diabetes classification

param_grid_diab = [
    {
        "classifier": [LogisticRegression(solver="liblinear", max_iter=2000)],
        "classifier__C": [0.001, 0.01, 0.1, 1, 10],
        "classifier__penalty": ["l2"],
        "classifier__class_weight": [None, "balanced"],
    },
    {
        "classifier": [RandomForestClassifier(random_state=42)],
        "classifier__n_estimators": [100, 300, 500],
        "classifier__max_depth": [None, 10, 20],
        "classifier__class_weight": [None, "balanced"],
    },
]

gs_diab = GridSearchCV(
    estimator=Pipeline([
        ("preprocessor", diab_preprocessor),
        ("classifier", RandomForestClassifier())  # placeholder
    ]),
    param_grid=param_grid_diab,
    cv=cv_strat_diab,
    scoring="balanced_accuracy",
    n_jobs=-1,
    verbose=1,
)

gs_diab.fit(X_train_diab, y_train_diab)

print("\nBest parameters from GridSearch (Diabetes LR+RF):")
print(gs_diab.best_params_)
print(f"Best mean CV balanced accuracy: {gs_diab.best_score_:.4f}\n")

best_model_diab = gs_diab.best_estimator_

In [ ]:
# Evaluate tuned model (this model may be LR or RF depending on grid)

y_diab_pred_gs, diab_gs_metrics = evaluate_classifier(
    best_model_diab,
    X_train_diab, 
    X_test_diab, 
    y_train_diab,
    y_test_diab, 
    target_name="Diabetes (Tuned)")

In [ ]:
# SMOTE + Random Undersampling 
imb_pipe_diab = ImbPipeline([
    ("preprocessor", diab_preprocessor),
    ("oversample", SMOTE(random_state=42)),
    ("undersample", RandomUnderSampler(random_state=42)),
    ("classifier", LogisticRegression(
        solver="liblinear", 
        penalty="l2",
        max_iter=2000,
        random_state=42
)),
])

scores_imb_diab = cross_val_score(imb_pipe_diab, X_train_diab, y_train_diab, cv=cv_strat_diab, scoring="balanced_accuracy", n_jobs=-1)
print(f'SMOTE+Under – balanced_accuracy MEAN: {scores_imb_diab.mean()} St Dev.:  {scores_imb_diab.std()}')

In [ ]:
y_oof_imb_diab = cross_val_predict(imb_pipe_diab, X_train_diab, y_train_diab, cv=cv_strat_diab, n_jobs=-1)
print("SMOTE+Under CV aggregated report (train):")
print(classification_report(y_train_diab, y_oof_imb_diab))
print("Confusion matrix (train CV):\n", confusion_matrix(y_train_diab, y_oof_imb_diab))
evaluate_classifier(imb_pipe_diab, X_train_diab, X_test_diab, y_train_diab, y_test_diab, target_name="Diabetes (SMOTE+Under)")

In [ ]:
# Fit + evaluate on test for the imbalanced pipeline
imb_pipe_diab.fit(X_train_diab, y_train_diab)
y_diab_test_imb = imb_pipe_diab.predict(X_test_diab)
print("SMOTE+Under Test report:")
print(classification_report(y_test_diab, y_diab_test_imb))
print("Confusion matrix (test):\n", confusion_matrix(y_test_diab, y_diab_test_imb))

In [ ]:
    # Create pretty labels
pretty_names = {
    "num__Age": "Age",
    "num__Gender": "Gender",
    "num__BMI": "BMI",
    "num__Glucose_Level": "Glucose Level",
    "num__Oral_Glucose_Test": "Oral Glucose Test",
    "num__Physical_Activity": "Physical Activity"
}


# Plot the logistic regression coefficients
if isinstance(best_model_diab.named_steps['classifier'], LogisticRegression):

    # Get feature names after preprocessing
    feature_names_diab = best_model_diab[:-1].get_feature_names_out()

    # Get coefficients
    coefs = best_model_diab.named_steps['classifier'].coef_[0]


    # Create DataFrame
    coef_df = pd.DataFrame({
        'Feature': feature_names_diab,
        'Coefficient': coefs
    })

    # Apply readable names
    coef_df['Feature'] = coef_df['Feature'].replace(pretty_names)

    # Sort by absolute value
    coef_df = coef_df.sort_values(by='Coefficient', key=abs, ascending=False)

    # Plot
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Coefficient', y='Feature', data=coef_df, palette='viridis')
    plt.title('Logistic Regression Coefficients for Diabetes Prediction', fontsize=16, fontweight='bold')
    plt.xlabel('Coefficient Value', fontsize=12)
    plt.ylabel('Feature', fontsize=12)
    plt.tight_layout()

    # Save high-resolution figure
    filename = "logistic_regression_coefficients_diabetes_300dpi.png"
    plt.savefig(FIGURE_DIR / filename, dpi=300, bbox_inches='tight')
    print(f"✅ Saved logistic regression coefficients plot as: {filename}")

    plt.show()


In [ ]:
# Get feature names after preprocessing
feature_names_diab = best_model_diab[:-1].get_feature_names_out()

result = permutation_importance(
    best_model_diab, X_test_diab, y_test_diab,
    n_repeats=10, random_state=42, n_jobs=2, scoring="balanced_accuracy"
)

sorted_idx = result.importances_mean.argsort()

importances_diab = pd.DataFrame(
    result.importances[sorted_idx].T,
    columns=feature_names_diab[sorted_idx]
)

importances_diab.rename(columns=pretty_names, inplace=True)

ax_diab = importances_diab.plot.box(vert=False, whis=10)
ax_diab.set_title("Permutation Importances (test set)")
ax_diab.axvline(x=0, color="k", linestyle="--")
ax_diab.set_xlabel("Decrease in balanced accuracy")
ax_diab.figure.tight_layout()

# Save the figure at 300 DPI
plt.savefig(FIGURE_DIR / "permutation_importances_diabetes.png", dpi=300, bbox_inches='tight')

fig = plt.gcf()
display(fig)
plt.show()

In [ ]:
X_test_transformed_diab = best_model_diab.named_steps['preprocessor'].transform(X_test_diab)
X_test_transformed_diab

In [ ]:
# Optional: install requirements-interpretation.txt before this section.
import shap
if not isinstance(best_model_diab.named_steps["classifier"], LogisticRegression):
    raise TypeError("This interpretation section is for logistic regression; adapt it for another selected model.")
X_train_transformed_diab = best_model_diab.named_steps["preprocessor"].transform(X_train_diab)
explainer_diab = shap.LinearExplainer(best_model_diab.named_steps["classifier"], X_train_transformed_diab)
explanation_full = explainer_diab(X_test_transformed_diab)
shap_values_diab = explanation_full.values


In [ ]:
# Plot the shap summary plot

feature_names_pretty = [pretty_names.get(f, f) for f in feature_names_diab]

shap.summary_plot(
    shap_values_diab,
    X_test_transformed_diab,
    feature_names=feature_names_pretty, show=False
)
plt.savefig(FIGURE_DIR / "shap_summary_plot_diabetes.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# If you used preprocessing (e.g., ColumnTransformer), get feature names like this:
if hasattr(best_model_diab[:-1], 'get_feature_names_out'):
    feature_names_diab = best_model_diab[:-1].get_feature_names_out()
else:
    # Fallback: default names if unknown
    n_features = X_test_transformed_diab.shape[1]
    feature_names_diab = [f"Feature_{i}" for i in range(n_features)]

In [ ]:
# Single-sample Waterfall Plot
# Single-sample Waterfall Plot
sample_index = 4

explanation_full = explainer_diab(X_test_transformed_diab)  # recompute full Explanation object

feature_names_pretty = [pretty_names.get(f, f) for f in feature_names_diab]

explanation = shap.Explanation(
    values=explanation_full.values[sample_index, :],
    base_values=explanation_full.base_values[sample_index],
    data=explanation_full.data[sample_index, :],
    feature_names=feature_names_pretty
)


shap.waterfall_plot(explanation, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / f"shap_waterfall_sample_{sample_index}_300dpi.png",
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Interpretation must be based on a clean execution of this revised notebook.
# Historical results and their limitations are documented in docs/FINDINGS.md.
# Target: reported diabetes/borderline status, not future onset.
# Grid search selects within training CV; do not choose a model using test results.
# The resampling classifier in cell 23 is logistic regression.
# No clinical, causal or population-level inference is established.
